# Week 13: Class database setup (instructor only)

Run this notebook **once** to load the two course datasets into the hosted
Postgres database on Neon, then check that the read-only student login works.

| Neon object | Value |
|---|---|
| Project | `ppol5203` (`late-mud-27069718`) |
| Branch | `production` |
| Database | `ppol5203_classdb` |
| Tables | `caseinit` (from `datasets/init_fordb.csv`), `divert` (from `datasets/divert.csv`) |
| Student login | `student`: read-only, 60-second query timeout |

The read-only student connection string is in `creds.yaml` (gitignored). The
owner string is entered when you run the notebook and is never saved or printed:
notebook output is stored in the `.ipynb` and pushed to GitHub with the website.

Re-running is safe: the tables are replaced each time.

In [7]:
# Install the Postgres driver into this notebook's Python if it's missing
import importlib.util, subprocess, sys
for pkg, pip_name in [("sqlalchemy", "sqlalchemy"), ("psycopg2", "psycopg2-binary")]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

from pathlib import Path
import os

import pandas as pd

## 1. Get the connection strings

* **Student (read-only)** string: read from `creds.yaml` → `class_database: connection_string`
  (the same place the slides and student activity read it from).
* **Owner** string: needed only here, to create the tables. The next cell uses
  `DATABASE_URL` from a `.env.local` file if you have one; otherwise it opens a
  hidden input box. Paste the string from the Neon console: project **ppol5203** →
  **Connect** → database `ppol5203_classdb`, role `neondb_owner`. It is not saved anywhere.

In [8]:
import yaml
from getpass import getpass

# Course root = the folder that holds datasets/ and creds.yaml
COURSE_ROOT = next(f for f in [Path.cwd(), *Path.cwd().parents] if (f / "datasets").is_dir())
DATASETS = COURSE_ROOT / "datasets"

with open(COURSE_ROOT / "creds.yaml") as stream:
    creds = yaml.safe_load(stream)
STUDENT_URL = creds["class_database"]["connection_string"]

# Owner string: .env.local if present, otherwise prompt (never printed)
OWNER_URL = os.environ.get("DATABASE_URL")
env_file = COURSE_ROOT / ".env.local"
if not OWNER_URL and env_file.exists():
    for line in env_file.read_text().splitlines():
        if line.startswith("DATABASE_URL="):
            OWNER_URL = line.split("=", 1)[1].strip().strip('"').strip("'")
if not OWNER_URL:
    OWNER_URL = getpass("Paste the Neon OWNER connection string for ppol5203_classdb: ").strip()

print("Course folder:", COURSE_ROOT)
print("Student string found:", STUDENT_URL.startswith("postgresql://student:"))
print("Owner string entered:", OWNER_URL.startswith("postgresql://"))

Paste the Neon OWNER connection string for ppol5203_classdb: ········
Course folder: /Users/rj545/Dropbox/PPOL5203_fall2026
Student string found: True
Owner string entered: True


## 2. Load the CSVs

In [9]:
TABLES = {
    "caseinit": "init_fordb.csv",
    "divert": "divert.csv",
}
DATE_COLS = ["received_date", "referral_date", "diversion_closed_date"]

def load_csv(filename):
    df = pd.read_csv(DATASETS / filename)
    # Lowercase column names so students never need "double quotes" in SQL
    df.columns = [c.lower() for c in df.columns]
    for col in DATE_COLS:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], format="%m/%d/%Y %I:%M:%S %p", errors="coerce")
    return df

frames = {table: load_csv(f) for table, f in TABLES.items()}
for table, df in frames.items():
    print(f"{table}: {df.shape[0]:,} rows x {df.shape[1]} columns")

caseinit: 62,056 rows x 8 columns
divert: 24,670 rows x 14 columns


## 3. Write the tables to Neon (as the owner)

This uses the Postgres driver (`psycopg2`) and Postgres's `COPY` command directly,
rather than `df.to_sql()`, so it works with any pandas version and loads in seconds.

In [10]:
import io
import psycopg2

def pg_type(dtype):
    s = str(dtype)
    if s.startswith("datetime64"):
        return "timestamp"
    return {"int64": "bigint", "float64": "double precision", "bool": "boolean"}.get(s, "text")

conn = psycopg2.connect(OWNER_URL)
with conn, conn.cursor() as cur:          # one transaction: all or nothing
    for table, df in frames.items():
        cols = ", ".join(f"{c} {pg_type(t)}" for c, t in df.dtypes.items())
        cur.execute(f"DROP TABLE IF EXISTS {table}")
        cur.execute(f"CREATE TABLE {table} ({cols})")
        buf = io.StringIO()
        df.to_csv(buf, index=False, header=False)   # blanks become NULL
        buf.seek(0)
        cur.copy_expert(f"COPY {table} FROM STDIN WITH (FORMAT csv)", buf)
        print(f"wrote {table}: {len(df):,} rows")
    # make sure the student login can read the (re)created tables
    cur.execute("GRANT USAGE ON SCHEMA public TO student")
    cur.execute("GRANT SELECT ON ALL TABLES IN SCHEMA public TO student")
conn.close()

wrote caseinit: 62,056 rows
wrote divert: 24,670 rows


## 4. Check everything as a student would see it

In [11]:
sconn = psycopg2.connect(STUDENT_URL)
with sconn.cursor() as cur:
    for table in TABLES:
        cur.execute(f"SELECT COUNT(*) FROM {table}")
        print(f"{table}: {cur.fetchone()[0]:,} rows visible to the student login")
    cur.execute("SELECT * FROM caseinit LIMIT 5")
    preview = pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])
sconn.rollback()
preview

caseinit: 62,056 rows visible to the student login
divert: 24,670 rows visible to the student login


,case_id,case_participant_id,offense_category,updated_offense_category,race,gender,age_at_incident,is_in_diversion
0,588837642829,442420671352,Sex Crimes,Human Trafficking,Black,Male,26.0,False
1,580726027874,432312003079,Narcotics,Narcotics,White [Hispanic or Latino],Male,27.0,True
2,508318872311,350031230404,UUW - Unlawful Use of Weapon,UUW - Unlawful Use of Weapon,Black,Male,29.0,True
3,565190677205,414462877324,Armed Robbery,Armed Robbery,Black,Male,31.0,False
4,570518536922,420717727182,DUI,Aggravated DUI,White [Hispanic or Latino],Male,41.0,True


In [12]:
# The student login should NOT be able to change anything
# (if it somehow could, the rollback undoes the delete)
try:
    with sconn.cursor() as cur:
        cur.execute("DELETE FROM divert")
    sconn.rollback()
    print("PROBLEM: the student login was able to delete rows")
except psycopg2.Error as e:
    sconn.rollback()
    print("Good: write blocked for student login ->", e.pgerror.splitlines()[0])
sconn.close()

Good: write blocked for student login -> ERROR:  permission denied for table divert


### Check your pandas version for the slides

The slides and student activity use `pd.read_sql_query(query, engine)` with SQLAlchemy 2.
That combination needs **pandas 2.2 or newer**; older pandas gives
`AttributeError: 'Engine' object has no attribute 'cursor'`.

In [13]:
import sqlalchemy
print("pandas", pd.__version__, "| sqlalchemy", sqlalchemy.__version__)
pd_ver = tuple(int(x) for x in pd.__version__.split(".")[:2])
sa_major = int(sqlalchemy.__version__.split(".")[0])
if sa_major >= 2 and pd_ver < (2, 2):
    print("Upgrade needed for the slides: run  %pip install -U pandas  then restart the kernel.")
else:
    print("OK: this pandas/SQLAlchemy combination works with pd.read_sql_query(query, engine).")

pandas 2.3.3 | sqlalchemy 1.4.39
OK: this pandas/SQLAlchemy combination works with pd.read_sql_query(query, engine).


## 5. What students run

Give students the `class_database: connection_string` value from `creds.yaml` (e.g. on Canvas,
not on the public course website). It is read-only, so the worst case if it
leaks is that someone reads these two tables.

```python
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine("PASTE_THE_CLASS_CONNECTION_STRING_HERE")

pd.read_sql("""
    SELECT race, COUNT(*) AS n, AVG(age_at_incident) AS mean_age
    FROM caseinit
    GROUP BY race
    ORDER BY n DESC
""", engine)
```

The first query after the database has been idle takes a few seconds while it
wakes up.